In [ ]:
# ============================================================
# CREDIT RISK PREDICTION
# File: model_training.py
#
# Purpose:
# - Load and prepare data
# - Train multiple classification models
# - Perform cross-validation
# - Tune hyperparameters
# - Compare models
# - Save the best model
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import joblib
import pandas as pd

from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)


# ============================================================
# 2. IMPORT OUR PREPROCESSING CODE
# ============================================================

from data_eda import df

from preprocessing import (
    prepare_data,
    preprocessor
)


# ============================================================
# 3. CREATE MODEL DIRECTORY
# ============================================================

os.makedirs(
    "models",
    exist_ok=True
)


# ============================================================
# 4. PREPARE DATA
# ============================================================

(
    X_train,
    X_test,
    y_train,
    y_test
) = prepare_data(df)


print("=" * 60)
print("DATA PREPARATION")
print("=" * 60)

print("\nTraining data shape:")
print(X_train.shape)

print("\nTesting data shape:")
print(X_test.shape)


# ============================================================
# 5. CREATE CROSS-VALIDATION STRATEGY
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ============================================================
# WHY STRATIFIED K-FOLD?
# ============================================================
#
# The target contains two classes:
#
# 0 → Good Risk
# 1 → Bad Risk
#
# StratifiedKFold attempts to preserve the class
# proportions in every fold.
#
# ============================================================


# ============================================================
# 6. LOGISTIC REGRESSION
# ============================================================

logistic_pipeline = Pipeline(
    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


# ------------------------------------------------------------
# Hyperparameters to test
# ------------------------------------------------------------

logistic_parameters = {

    "model__C": [
        0.01,
        0.1,
        1,
        10
    ]
}


# ------------------------------------------------------------
# Grid Search
# ------------------------------------------------------------

logistic_grid = GridSearchCV(

    estimator=logistic_pipeline,

    param_grid=logistic_parameters,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1
)


print("\n" + "=" * 60)
print("TRAINING LOGISTIC REGRESSION")
print("=" * 60)


logistic_grid.fit(
    X_train,
    y_train
)


print(
    "\nBest Logistic Regression parameters:"
)

print(
    logistic_grid.best_params_
)


print(
    "\nBest Cross-Validation ROC-AUC:"
)

print(
    round(
        logistic_grid.best_score_,
        4
    )
)


# ============================================================
# 7. RANDOM FOREST
# ============================================================

random_forest_pipeline = Pipeline(
    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            RandomForestClassifier(
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


# ------------------------------------------------------------
# Hyperparameters to test
# ------------------------------------------------------------

random_forest_parameters = {

    "model__n_estimators": [
        100,
        200
    ],

    "model__max_depth": [
        None,
        5,
        10
    ],

    "model__min_samples_split": [
        2,
        5
    ]
}


# ------------------------------------------------------------
# Grid Search
# ------------------------------------------------------------

random_forest_grid = GridSearchCV(

    estimator=random_forest_pipeline,

    param_grid=random_forest_parameters,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1
)


print("\n" + "=" * 60)
print("TRAINING RANDOM FOREST")
print("=" * 60)


random_forest_grid.fit(
    X_train,
    y_train
)


print(
    "\nBest Random Forest parameters:"
)

print(
    random_forest_grid.best_params_
)


print(
    "\nBest Cross-Validation ROC-AUC:"
)

print(
    round(
        random_forest_grid.best_score_,
        4
    )
)


# ============================================================
# 8. GRADIENT BOOSTING
# ============================================================

gradient_boosting_pipeline = Pipeline(
    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            GradientBoostingClassifier(
                random_state=42
            )
        )
    ]
)


# ------------------------------------------------------------
# Hyperparameters to test
# ------------------------------------------------------------

gradient_boosting_parameters = {

    "model__n_estimators": [
        100,
        200
    ],

    "model__learning_rate": [
        0.03,
        0.05,
        0.1
    ],

    "model__max_depth": [
        2,
        3
    ]
}


# ------------------------------------------------------------
# Grid Search
# ------------------------------------------------------------

gradient_boosting_grid = GridSearchCV(

    estimator=gradient_boosting_pipeline,

    param_grid=gradient_boosting_parameters,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1
)


print("\n" + "=" * 60)
print("TRAINING GRADIENT BOOSTING")
print("=" * 60)


gradient_boosting_grid.fit(
    X_train,
    y_train
)


print(
    "\nBest Gradient Boosting parameters:"
)

print(
    gradient_boosting_grid.best_params_
)


print(
    "\nBest Cross-Validation ROC-AUC:"
)

print(
    round(
        gradient_boosting_grid.best_score_,
        4
    )
)


# ============================================================
# 9. COLLECT TRAINED MODELS
# ============================================================

models = {

    "Logistic Regression":
        logistic_grid.best_estimator_,

    "Random Forest":
        random_forest_grid.best_estimator_,

    "Gradient Boosting":
        gradient_boosting_grid.best_estimator_
}


# ============================================================
# 10. EVALUATE MODELS ON TEST SET
# ============================================================

results = []


for model_name, model in models.items():

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    y_pred = model.predict(
        X_test
    )


    # --------------------------------------------------------
    # Probability of Bad Risk
    # --------------------------------------------------------

    y_probability = model.predict_proba(
        X_test
    )[:, 1]


    # --------------------------------------------------------
    # Evaluation metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )


    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )


    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )


    roc_auc = roc_auc_score(
        y_test,
        y_probability
    )


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    results.append({

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1_Score": f1,

        "ROC_AUC": roc_auc
    })


# ============================================================
# 11. MODEL COMPARISON TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)


results_df = results_df.sort_values(
    by="ROC_AUC",
    ascending=False
)


print("\n" + "=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 12. SAVE MODEL RESULTS
# ============================================================

results_df.to_csv(
    "outputs/model_results.csv",
    index=False
)


print(
    "\nModel comparison saved to:"
)

print(
    "outputs/model_results.csv"
)


# ============================================================
# 13. SELECT BEST MODEL
# ============================================================

best_model_name = results_df.iloc[0]["Model"]


best_model = models[
    best_model_name
]


print("\n" + "=" * 60)
print("BEST MODEL")
print("=" * 60)

print(
    "Best model:",
    best_model_name
)


print(
    "ROC-AUC:",
    round(
        results_df.iloc[0]["ROC_AUC"],
        4
    )
)


# ============================================================
# 14. SAVE BEST MODEL
# ============================================================

model_path = (
    "models/best_model.pkl"
)


joblib.dump(
    best_model,
    model_path
)


print(
    "\nBest model saved successfully:"
)

print(
    model_path
)


# ============================================================
# 15. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 60)

print(
    "MODEL TRAINING COMPLETED"
)

print("=" * 60)